In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '1_train_full.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '1_test.parquet')

## 1. 데이터 피쳐링(파생변수 생성)

In [2]:
# ============================================================
# 1-1. 피쳐링: 수위 차분 파생변수 생성
# ============================================================
# wl_diff_00h = wl_t(현재수위) - wl_t_minus_00h 파생변수 5개 생성

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)

for h in [1, 3, 6, 12, 24]:
    train_full[f"wl_diff_{h}h"] = (
        train_full['wl_t'] - train_full[f"wl_t_minus_{h}h"]
    )
    test[f"wl_diff_{h}h"] = (
        test['wl_t'] - test[f"wl_t_minus_{h}h"]
    )

print('train 수위차분 파생변수 추가 후 크기: ', train_full.shape)
print('test 수위차분 파생변수 추가 후 크기: ', test.shape)


train 파생변수 추가 전 크기:  (749657, 25)
test 파생변수 추가 전 크기:  (186263, 24)
train 수위차분 파생변수 추가 후 크기:  (749657, 30)
test 수위차분 파생변수 추가 후 크기:  (186263, 29)


In [3]:
# ============================================================
# 1-2. 피쳐링: 강우강도 파생변수 생성
# ============================================================
# imerg_int_0to0h = (imerg_rain_00h - imerg_rain_00h)/hours
# sfc_int_0to0h = (sfc_rain_00h - sfc_rain_00h)/hours

print('train 파생변수 추가 전 크기: ', train_full.shape)
print('test 파생변수 추가 전 크기: ', test.shape)


for df in [train_full, test]:

    df['imerg_int_1to6h'] = (df['imerg_rain_6h'] - df['imerg_rain_1h']) / 5
    df['imerg_int_6to24h'] = (df['imerg_rain_24h'] - df['imerg_rain_6h']) / 18

    df['sfc_int_1to6h'] = (df['sfc_rain_6h'] - df['sfc_rain_1h']) / 5
    df['sfc_int_6to24h'] = (df['sfc_rain_24h'] - df['sfc_rain_6h']) / 18


print('train 강우강도 파생변수 추가 후 크기: ', train_full.shape)
print('test 강우강도 파생변수 추가 후 크기: ', test.shape)

train 파생변수 추가 전 크기:  (749657, 30)
test 파생변수 추가 전 크기:  (186263, 29)
train 강우강도 파생변수 추가 후 크기:  (749657, 34)
test 강우강도 파생변수 추가 후 크기:  (186263, 33)


In [ ]:
# ============================================================
# 1-3. 피쳐링 결과: 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 독립변수군
feature_cols = [
    col for col in train_full.columns
    if col not in ('timestamp', target_col)
]

assert len(feature_cols) == 32, (
    f'독립변수 개수 이상: 예상 32개, 현재 {len(feature_cols)}개'
)

## 2. 이상치 제거

In [6]:
# ============================================================
# 2-1. 이상치(계측오류) 제거: train_full 종속변수 이상치 1개 제거
# ============================================================
# 변수별 분포특성 확인단계에서 확인됨

# 제거 조건 설정
cond_maxrise_outlier = train_full['target_maxrise_6h'] < -8  # 이거 자체가 불리언 조건문임
print(f" cond_maxrise_outlier 조건에 해당하는 데이터 개수: {cond_maxrise_outlier.sum()}")  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
maxrise_outlier_row_id = (
    train_full
    .loc[cond_maxrise_outlier]
    .index
)
print(maxrise_outlier_row_id)

# 이상치 제거
train_full = train_full.drop(index = maxrise_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()

 cond_maxrise_outlier 조건에 해당하는 데이터 개수: 1
이상치 행 제거 전 데이터 개수: 749657

Index(['TR_000583385'], dtype='object', name='row_id')
이상치 행 제거 후 train셋 행 개수: 749656



In [8]:
# ============================================================
# 2-2. 이상치(계측오류) 제거: 수위차분변수-종속변수 산점도 이상치
# ============================================================
# 변수간 상관계수 분석에서 확인됨

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 제거 조건 설정
cond_wl_out_ST1 = train_full['station_id'] == 'ST0445'
cond_wl_out_ST2 = train_full['station_id'] == 'ST0066'
cond_wl_out_ST0445 = (train_full[wl_cols] > 15).any(axis=1)
cond_wl_out_ST0066 = (train_full[wl_cols] > 23).any(axis=1)
cond_wl_f = (
    (cond_wl_out_ST1 & cond_wl_out_ST0445)|
    (cond_wl_out_ST2 & cond_wl_out_ST0066)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
wl_outlier_row_id = (
    train_full
    .loc[cond_wl_f]
    .index
)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()

ccond_wl_f 조건에 해당하는 데이터 개수: 32
이상치 행 제거 전 데이터 개수: 749656

이상치 행 제거 후 train셋 행 개수: 749624



In [10]:
# ============================================================
# 2-3. 이상치(계측오류) 제거: 수위차분변수 이상치 제거
# ============================================================
# train, test셋 분포 비교에서 확인

# 제거 조건 설정
cond_wl_out_ST3 = train_full['station_id'] == 'ST0089'
cond_wl_out_ST4 = train_full['station_id'] == 'ST0158'
cond_wl_out_ST5 = train_full['station_id'] == 'ST0448'
cond_wl_out_ST6 = train_full['station_id'] == 'ST0007'
cond_wl_out_ST0089 = (train_full[wl_cols] > 6).any(axis=1)
cond_wl_out_ST0158 = (train_full[wl_cols] > 7).any(axis=1)
cond_wl_out_ST0448 = (
    (train_full[wl_cols] > 9).any(axis=1) &
    (train_full.index != 'TR_000199803')
)
cond_wl_out_ST0007 = (train_full[wl_cols] > 8).any(axis=1)
cond_wl_f2 = (
    (cond_wl_out_ST3 & cond_wl_out_ST0089)|
    (cond_wl_out_ST4 & cond_wl_out_ST0158)|
    (cond_wl_out_ST5 & cond_wl_out_ST0448)|
    (cond_wl_out_ST6 & cond_wl_out_ST0007)
    )

print(
    f"ccond_wl_f 조건에 해당하는 데이터 개수: "
    f"{cond_wl_f2.sum()}"
)  # 불리언조건문은 sum으로 개수를 셀 수 있음
print(f"이상치 행 제거 전 데이터 개수: {len(train_full)}") 

# 제거 대상 지정
print()
wl_outlier2_row_id = (
    train_full
    .loc[cond_wl_f2]
    .index
)
print(wl_outlier2_row_id)

# 이상치 제거
train_full = train_full.drop(index = wl_outlier2_row_id)
print(f"이상치 행 제거 후 train셋 행 개수: {len(train_full)}")
print()


ccond_wl_f 조건에 해당하는 데이터 개수: 15
이상치 행 제거 전 데이터 개수: 749624

Index(['TR_000547855', 'TR_000457689', 'TR_000318564', 'TR_000193482',
       'TR_000057205', 'TR_000154692', 'TR_000715299', 'TR_000468355',
       'TR_000450715', 'TR_000267036', 'TR_000476408', 'TR_000185855',
       'TR_000385460', 'TR_000665570', 'TR_000480061'],
      dtype='object', name='row_id')
이상치 행 제거 후 train셋 행 개수: 749609



In [ ]:
# ============================================================
# 2-4. 최종 train_full, test 저장
# ============================================================

train_full.to_parquet(PROJECT_ROOT / 'data' / '1_train_full.parquet')
test.to_parquet(PROJECT_ROOT / 'data' / '1_test_fe.parquet')

## 3. 강우사상 라벨링, GroupKFold 5-Fold 배정(CSV로 저장)

In [14]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분: diff 계산
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 붙이기(매핑)
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙(조인대상 인덱스 필수)

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

In [ ]:
# ============================================================
# 3-2. GroupKFold 5-Fold 배정, csv 저장
# ============================================================


In [18]:
print(train_full.shape)

assert train_full.shape[1] == 35, '컬럼개수 이상'

(749609, 35)


## 4. Tidal, Normal 데이터셋 분할

## 5. Tidal 데이터셋 모델 생성: 단일 홀드아웃, CatBoost, ExtraTree 앙상블

## 6. Normal 데이터셋 모델 생성: K_Fold 교차검증, CatBoost

## 7. Tidal / Normal 예측 결과 통합 및 제출파일 생성

## 4. Tidal, Normal 데이터셋 분할